In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import ast

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder


from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import train_test_split
import shap

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df_movies = pd.read_parquet("../data/movies_filtered_cleaned.parquet")
df_ratings = pd.read_parquet("../data/processed_df_ratings_filtered.parquet")
df_ratings = df_ratings[['userId', 'movieId', 'rating']]

df_users = pd.read_parquet("../data/all_users_stats_post_movies_filter.parquet")

In [ ]:
df_movies.columns

In [ ]:
df_movies.head(1)

In [ ]:
df_ratings.columns

In [ ]:
df_ratings.head(1)

In [ ]:
df_users.columns

In [ ]:
df_users.head(1)

In [ ]:
all_genres_from_list = set(g for genres in df_movies['genre_list'] for g in genres)

all_genres_from_main = set(df_movies['main_genre'].dropna().unique())

all_genres = all_genres_from_list | all_genres_from_main

genre2id = {genre: idx for idx, genre in enumerate(sorted(all_genres))}
id2genre = {idx: genre for genre, idx in genre2id.items()}

In [ ]:
df_movies['main_genre_id'] = df_movies['main_genre'].map(genre2id)

def get_genre_id_list(row):
    genre_ids = [genre2id[g] for g in row['genre_list'] if g in genre2id]
    if genre_ids:
        return genre_ids
    elif pd.notnull(row['main_genre']):
        return [genre2id[row['main_genre']]]
    else:
        return []
df_movies['genre_id_list'] = df_movies.apply(get_genre_id_list, axis=1)

In [ ]:
all_directors = df_movies['director'].dropna().unique()
director2id = {director: idx for idx, director in enumerate(sorted(all_directors))}
id2director = {idx: director for director, idx in director2id.items()}
df_movies['director_id'] = df_movies['director'].map(director2id)

all_actors = df_movies['lead_actor'].dropna().unique()
actor2id = {actor: idx for idx, actor in enumerate(sorted(all_actors))}
id2actor = {idx: actor for actor, idx in actor2id.items()}
df_movies['actor_id'] = df_movies['lead_actor'].map(actor2id)

In [ ]:
len(df_movies['actor_id'].unique())

In [ ]:
len(df_movies['lead_actor'].unique())


In [ ]:
df_movies

In [ ]:
def get_top_entities(df, group_col, entity_col, n=2):
    """
    Returns a list of the top N entities (e.g., directors or actors) for each group value.
    Example: top directors in a genre, or top actors for a director's movies, etc.
    """
    counts = (
        df.groupby([group_col, entity_col])
        .size()
        .reset_index(name='num_movies')
    )
    top_entities = (
        counts.sort_values(['num_movies', entity_col], ascending=[False, True])
        .groupby(group_col)
        .head(n)
        .reset_index(drop=True)
    )
    top_entities_pivot = (
        top_entities.groupby(group_col)[entity_col]
        .apply(list)
        .to_dict()
    )
    return top_entities_pivot


In [ ]:
def get_top_movie_id(df, group_col, sort_col='vote_average'):
    """
    Returns a dict {group: top_movie_id} for each group (e.g., genre).
    """
    idx = df.groupby(group_col)[sort_col].idxmax()
    top_movies = df.loc[idx, [group_col, 'movieId']].drop_duplicates(group_col)
    return dict(zip(top_movies[group_col], top_movies['movieId']))


In [ ]:
def get_top_entities_with_counts(df, group_col, entity_col, n=2):
    """
    For each unique value in group_col, returns a list of (entity, count) tuples
    for the top N entities with the highest count, sorted descending.
    Example: top directors (with movie counts) for each genre.
    """
    counts = (
        df.groupby([group_col, entity_col])
        .size()
        .reset_index(name='num_movies')
    )
    counts = counts.sort_values([group_col, 'num_movies', entity_col], ascending=[True, False, True])
    result = (
        counts.groupby(group_col)
        .apply(lambda group: list(zip(group[entity_col], group['num_movies']))[:n])
        .to_dict()
    )
    return result


# filtered_df that we use in the dashboard is from df_movies


In [ ]:
filtered_df = df_movies.copy()

selected_genre = 'Action'
selected_year_min = 1910
selected_year_max = 2017

filtered_df = filtered_df.explode('genre_list')

filtered_df = filtered_df[
    (filtered_df['genre_list'] == selected_genre) &
    (filtered_df['release_year'] >= selected_year_min) &
    (filtered_df['release_year'] <= selected_year_max)
]


In [ ]:
top_directors_by_genre = get_top_entities_with_counts(filtered_df, 'main_genre', 'director', n=2)

genre = 'Action'
top_directors = top_directors_by_genre.get(genre, [])

top_actors_by_genre = get_top_entities_with_counts(filtered_df, 'main_genre', 'lead_actor', n=2)
top_actors = top_actors_by_genre.get(genre, [])

top_movie_ids_by_genre = get_top_movie_id(filtered_df, 'main_genre', sort_col='vote_average')
top_movie_id = top_movie_ids_by_genre.get(genre, None)


In [ ]:
def get_main_genre(x):
    mode = x.mode()
    return mode.iloc[0] if not mode.empty else None

def get_sub_genre(x):
    mode = x.mode()
    return mode.iloc[1] if len(mode) > 1 else None

def get_df_entity(
    df,
    entity_col,
    popularity_col,
    avg_rating_col,
    main_genre_col='main_genre',
    sub_genre_col='main_genre',
    count_col='movieId',
    additional_aggs=None
):
    aggs = {
        f'{entity_col}_popularity': (popularity_col, 'first'),
        'num_movies': (count_col, 'count'),
        f'{entity_col}_main_genre': (main_genre_col, get_main_genre),
        f'{entity_col}_sub_genre': (sub_genre_col, get_sub_genre),
        f'{entity_col}_avg_rating': (avg_rating_col, 'mean'),
        'total_vote_count': ('vote_count', 'sum'),
        'avg_popularity_score': ('popularity_score', 'mean'),
        'avg_critical_success': ('critical_success', 'mean'),
    }
    if additional_aggs:
        aggs.update(additional_aggs)
    return df.groupby(entity_col).agg(**aggs).reset_index().sort_values([f'{entity_col}_popularity', "num_movies", "avg_popularity_score"], ascending=False)


def get_top_entities_with_counts(df, group_col, entity_col, n=2):
    counts = (
        df.groupby([group_col, entity_col])
        .size()
        .reset_index(name='num_movies')
    )
    counts = counts.sort_values([group_col, 'num_movies', entity_col], ascending=[True, False, True])
    result = (
        counts.groupby(group_col)
        .apply(lambda group: list(zip(group[entity_col], group['num_movies']))[:n])
        .to_dict()
    )
    return result

def get_df_genres(df):
    df_exploded = df.explode('genre_list').rename(columns={'genre_list': 'genre'})
    director_counts = df_exploded.groupby(['genre', 'director']).size().reset_index(name='num_movies')
    top2_directors = director_counts.groupby('genre').apply(
        lambda group: pd.Series({
            'top_director': group.sort_values('num_movies', ascending=False)['director'].iloc[0] if len(group) > 0 else None,
            'sub_director': group.sort_values('num_movies', ascending=False)['director'].iloc[1] if len(group) > 1 else None
        })
    ).reset_index()

    actor_counts = df_exploded.groupby(['genre', 'lead_actor']).size().reset_index(name='num_movies')
    top2_actors = actor_counts.groupby('genre').apply(
        lambda group: pd.Series({
            'top_lead_actor': group.sort_values('num_movies', ascending=False)['lead_actor'].iloc[0] if len(group) > 0 else None,
            'sub_lead_actor': group.sort_values('num_movies', ascending=False)['lead_actor'].iloc[1] if len(group) > 1 else None
        })
    ).reset_index()

    idx = df_exploded.groupby('genre')['vote_average'].idxmax()
    top_movie_ids = df_exploded.loc[idx, ['genre', 'movieId']].drop_duplicates('genre').set_index('genre')

    df_genres = (
        df_exploded.groupby('genre').agg(
            num_movies=('movieId', 'count'),
            avg_rating=('vote_average', 'mean'),
            total_vote_count=('vote_count', 'sum'),
            avg_popularity_score=('popularity_score', 'mean'),
            avg_critical_success=('critical_success', 'mean')
        ).reset_index()
        .merge(top2_directors, on='genre', how='left')
        .merge(top2_actors, on='genre', how='left')
    )
    df_genres = df_genres.merge(
        top_movie_ids[['movieId']], left_on='genre', right_index=True, how='left'
    ).rename(columns={'movieId': 'top_movie_id'})

    return df_genres


In [ ]:
filtered_df = df_movies.copy()

df_directors = get_df_entity(
    df=filtered_df,
    entity_col='director',
    popularity_col='director_popularity',
    avg_rating_col='vote_average'
)

df_lead_actors = get_df_entity(
    df=filtered_df,
    entity_col='lead_actor',
    popularity_col='lead_actor_popularity',
    avg_rating_col='vote_average'
)

df_genres = get_df_genres(filtered_df)


In [ ]:
len(df_lead_actors["lead_actor"].unique())

In [ ]:
df_lead_actors["num_movies"].value_counts()

In [ ]:
df_directors

In [ ]:
df_lead_actors

In [ ]:
STOP CODE

All code must use python pandas, numpy, plotly graph objects for graphs

# First I want to do all movie statistics.

1. Average movie ratings by genre
2. Top 10 directors by director popularity,and on hover we show number of movies they were in and their director_main_genre and director sub_genre - second most genre
3. Top lead actors by lead_actor_popularity and on hover we show the average ratings of the actor and number of movies they were in and their actor_main_genre and actor sub_genre - second most genre
4. The top 10 movies by popularity score and on hover we see number of votes, average rating, crowd_approval, critical success of the movie

I will be building a movie recommender system so things will need to be filtered n all.
So I want a few suggestions:

Do I create separate csv files for directors, and, lead actors with their calculated statistics so that when we do filters or should I add it to df movies itself?
Please provide code for the same.

And then provide code for the above four graphs.


In [ ]:
def get_main_genre(x):
    mode = x.mode()
    return mode.iloc[0] if not mode.empty else None

def get_sub_genre(x):
    mode = x.mode()
    return mode.iloc[1] if len(mode) > 1 else None

df_directors = (
    df_movies.groupby('director').agg(
        director_popularity=('director_popularity', 'first'),
        num_movies=('movieId', 'count'),
        director_main_genre=('main_genre', get_main_genre),
        director_sub_genre=('main_genre', get_sub_genre),
        avg_vote_average=('vote_average', 'mean'),
        total_vote_count=('vote_count', 'sum'),
        avg_popularity_score=('popularity_score', 'mean'),
        avg_critical_success=('critical_success', 'mean'),
    ).reset_index()
)

# --- LEAD ACTORS ---

df_lead_actors = (
    df_movies.groupby('lead_actor').agg(
        lead_actor_popularity=('lead_actor_popularity', 'first'),
        num_movies=('movieId', 'count'),
        actor_main_genre=('main_genre', get_main_genre),
        actor_sub_genre=('main_genre', get_sub_genre),
        actor_avg_rating=('vote_average', 'mean'),
        total_vote_count=('vote_count', 'sum'),
        avg_popularity_score=('popularity_score', 'mean'),
        avg_critical_success=('critical_success', 'mean'),
    ).reset_index()
)


In [ ]:
df_directors.to_parquet('../data/directors_stats.parquet', index=False)
df_lead_actors.to_parquet('../data/lead_actors_stats.parquet', index=False)

In [ ]:
df_exploded = df_movies.explode('genre_list').rename(columns={'genre_list': 'genre'})

def get_top2_entities(group, entity_col):
    top = group.sort_values('num_movies', ascending=False)
    entities = top[entity_col].tolist()
    return pd.Series({
        f'top_{entity_col}': entities[0] if len(entities) > 0 else None,
        f'sub_{entity_col}': entities[1] if len(entities) > 1 else None
    })

# Top directors
director_counts = df_exploded.groupby(['genre', 'director']).size().reset_index(name='num_movies')
top2_directors = director_counts.groupby('genre').apply(get_top2_entities, entity_col='director').reset_index()

# Top actors
actor_counts = df_exploded.groupby(['genre', 'lead_actor']).size().reset_index(name='num_movies')
top2_actors = actor_counts.groupby('genre').apply(get_top2_entities, entity_col='lead_actor').reset_index()

# Get top movie ID (highest-rated) per genre - ensuring one per genre
idx = df_exploded.groupby('genre')['vote_average'].idxmax()
top_movie_ids = df_exploded.loc[idx, ['genre', 'movieId']].drop_duplicates('genre').set_index('genre')

# The main df_genres
df_genres = (
    df_exploded.groupby('genre').agg(
        num_movies=('movieId', 'count'),
        avg_rating=('vote_average', 'mean'),
        total_vote_count=('vote_count', 'sum'),
        avg_popularity_score=('popularity_score', 'mean'),
        avg_critical_success=('critical_success', 'mean')
    ).reset_index()
    .merge(top2_directors, on='genre', how='left')
    .merge(top2_actors, on='genre', how='left')
)

# Merge top_movie_id (one per genre)
df_genres = df_genres.merge(
    top_movie_ids[['movieId']], left_on='genre', right_index=True, how='left'
).rename(columns={'movieId': 'top_movie_id'})

# df_genres.to_csv('genres_stats.csv', index=False)


In [ ]:
df_genres

In [ ]:
STOP CODE

In [ ]:
df_exploded = df_movies.explode('genre_list').rename(columns={'genre_list': 'genre'})

df_genres = (
    df_exploded.groupby('genre').agg(
        num_movies=('movieId', 'count'),
        avg_rating=('vote_average', 'mean'),
        total_vote_count=('vote_count', 'sum'),
        avg_popularity_score=('popularity_score', 'mean'),
        avg_critical_success=('critical_success', 'mean'),
    )
    .reset_index()
    .sort_values('num_movies', ascending=False)
)
df_genres

In [ ]:
def get_top2_entities(group, entity_col):
    top = group.sort_values('num_movies', ascending=False)
    entities = top[entity_col].tolist()
    return pd.Series({
        f'top_{entity_col}': entities[0] if len(entities) > 0 else None,
        f'sub_{entity_col}': entities[1] if len(entities) > 1 else None
    })


df_exploded = df_movies.explode('genre_list').rename(columns={'genre_list': 'genre'})

director_counts = (
    df_exploded.groupby(['genre', 'director'])
    .size()
    .reset_index(name='num_movies')
)

top_directors = (
    director_counts.sort_values(['genre', 'num_movies'], ascending=[True, False])
    .drop_duplicates('genre')
    .set_index('genre')
)


top2_directors = (
    director_counts.groupby('genre')
    .apply(get_top2_entities, entity_col='director')
    .reset_index()
)


actor_counts = (
    df_exploded.groupby(['genre', 'lead_actor'])
    .size()
    .reset_index(name='num_movies')
)

top_actors = (
    actor_counts.sort_values(['genre', 'num_movies'], ascending=[True, False])
    .drop_duplicates('genre')
    .set_index('genre')
)

top2_actors = (
    actor_counts.groupby('genre')
    .apply(get_top2_entities, entity_col='lead_actor')
    .reset_index()
)


df_genres = (
    df_exploded.groupby('genre').agg(
        num_movies=('movieId', 'count'),
        avg_rating=('vote_average', 'mean'),
        total_vote_count=('vote_count', 'sum'),
        avg_popularity_score=('popularity_score', 'mean'),
        avg_crowd_approval=('crowd_approval', 'mean')
    )
    .reset_index()
)

# Merge top director/actor info
df_genres = (
    df_genres
    .merge(top2_directors, on='genre', how='left')
    .merge(top2_actors, on='genre', how='left')
)

# Save to CSV if needed
df_genres.to_csv('genres_stats.csv', index=False)


In [ ]:
df_genres

In [ ]:
df_genres

In [ ]:
df_exploded = df_movies.explode('genre_list')

genre_ratings = (
    df_exploded.groupby('genre_list')['vote_average']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig = go.Figure([
    go.Bar(
        x=genre_ratings['genre_list'],
        y=genre_ratings['vote_average'],
        text=genre_ratings['vote_average'].round(2),
        hovertemplate="Genre: %{x}<br>Average Rating: %{y:.2f}<extra></extra>"
    )
])
fig.update_layout(
    title="Average Movie Ratings by Genre",
    xaxis_title="Genre",
    yaxis_title="Average Rating",
    xaxis_tickangle=45
)
fig.show()

In [ ]:
top_directors = df_directors.sort_values('director_popularity', ascending=False).head(10)

fig = go.Figure([
    go.Bar(
        x=top_directors['director'],
        y=top_directors['director_popularity'],
        hovertemplate=(
            "Director: %{x}<br>"
            "Popularity: %{y:.2f}<br>"
            "Number of Movies: %{customdata[0]}<br>"
            "Main Genre: %{customdata[1]}<br>"
            "Sub Genre: %{customdata[2]}<extra></extra>"
        ),
        customdata=top_directors[['num_movies', 'director_main_genre', 'director_sub_genre']].values
    )
])
fig.update_layout(
    title="Top 10 Directors by Popularity",
    xaxis_title="Director",
    yaxis_title="Popularity",
    xaxis_tickangle=45
)
fig.show()


In [ ]:
top_actors = df_lead_actors.sort_values('lead_actor_popularity', ascending=False).head(10)

fig = go.Figure([
    go.Bar(
        x=top_actors['lead_actor'],
        y=top_actors['lead_actor_popularity'],
        hovertemplate=(
            "Lead Actor: %{x}<br>"
            "Popularity: %{y:.2f}<br>"
            "Avg Rating: %{customdata[0]:.2f}<br>"
            "Number of Movies: %{customdata[1]}<br>"
            "Main Genre: %{customdata[2]}<br>"
            "Sub Genre: %{customdata[3]}<extra></extra>"
        ),
        customdata=top_actors[['actor_avg_rating', 'num_movies', 'actor_main_genre', 'actor_sub_genre']].values
    )
])
fig.update_layout(
    title="Top 10 Lead Actors by Popularity",
    xaxis_title="Lead Actor",
    yaxis_title="Popularity",
    xaxis_tickangle=45
)
fig.show()


In [ ]:
top_movies = df_movies.sort_values('popularity_score', ascending=False).head(10)

fig = go.Figure([
    go.Bar(
        x=top_movies['title'],
        y=top_movies['popularity_score'],
        hovertemplate=(
            "Movie: %{x}<br>"
            "Popularity Score: %{y:.2f}<br>"
            "Votes: %{customdata[0]}<br>"
            "Avg Rating: %{customdata[1]:.2f}<br>"
            "Crowd Approval: %{customdata[2]:.2f}<br>"
            "Critical Success: %{customdata[3]:.2f}<extra></extra>"
        ),
        customdata=top_movies[['vote_count', 'vote_average', 'crowd_approval', 'critical_success']].values
    )
])
fig.update_layout(
    title="Top 10 Movies by Popularity Score",
    xaxis_title="Movie",
    yaxis_title="Popularity Score",
    xaxis_tickangle=45
)
fig.show()


In [ ]:
def describe_cluster(cluster_movies, n_genres=3, n_topics=3):
    main_genre_cols = [col for col in cluster_movies.columns if col.startswith('main_genre_')]
    top_genres = cluster_movies[main_genre_cols].sum().sort_values(ascending=False).head(n_genres)
    genres_desc = ', '.join([g.replace('main_genre_', '') for g in top_genres.index])
    topic_cols = [col for col in cluster_movies.columns if col.startswith('topic_')]
    topic_means = cluster_movies[topic_cols].mean().sort_values(ascending=False)
    top_topics = [(int(i.split('_')[1]), topic_means[i]) for i in topic_means.head(n_topics).index]
    topic_desc = []
    for topic_num, _ in top_topics:
        top_words = ', '.join([feature_names[i] for i in nmf.components_[topic_num].argsort()[-4:][::-1]])
        topic_desc.append(f"Topic {topic_num}: {top_words}")
    top_decades = cluster_movies['release_decade'].value_counts().head(2)
    decades_desc = ', '.join(map(str, top_decades.index))

    top_year = cluster_movies['release_decade'].value_counts().head(2)
    year_desc = ', '.join(map(str, top_year.index))
    example_titles = ', '.join(cluster_movies['title'].head(3))
    return f"Genres: {genres_desc} | Decades: {decades_desc} | Year: {year_desc} | Top topics: {'; '.join(topic_desc)} | Example movies: {example_titles}"
